# 🔍 Chapter 3: Unsupervised Learning and Preprocessing
**Book Reference:** *Introduction to Machine Learning with Python: A Guide for Data Scientists* by Andreas C. Müller & Sarah Guido

---

## 3.1 Introduction to Unsupervised Learning
Unlike supervised learning, where the model is guided by an explicit target vector ($y$), **Unsupervised Learning** processes inputs ($X$) without any corresponding labels. The goal is not to predict an output, but rather to extract structural patterns, discover natural groupings, or create more compact, efficient representations of the underlying data.

The book outlines two main categories of unsupervised learning algorithms:
1. **Data Transformations:** Algorithms that alter the feature space to make it easier for humans to visualize or more efficient for other machine learning models to ingest. Examples include scaling, normalization, and dimensionality reduction.
2. **Clustering:** Algorithms that partition data points into distinct groups (clusters) based on feature similarity (e.g., segmenting customer demographics or grouping similar text documents).

Evaluating unsupervised models is inherently challenging. Because there is no "ground truth" label to score against, success is often judged qualitatively, via visualization, or by measuring downstream performance improvements in supervised models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer, make_blobs, make_moons
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Set uniform environment seeding and visualization patterns
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

print("Libraries successfully imported for Chapter 3 Unsupervised Learning workflow.")

## 3.2 Preprocessing and Scaling
Before diving into complex unsupervised transformations like PCA or clustering, data must undergo fundamental numerical scaling. Many machine learning estimators (such as Support Vector Machines, Neural Networks, and distance-based clustering algorithms like K-Means) assume that all features are centered around zero and share a similar variance scale.

### 3.2.1 Types of Scalers
The book highlights four foundational scaling transformers natively available in `scikit-learn`:

* **`StandardScaler`:** Standardizes features by shifting the mean to 0 and scaling the variance to 1. 
  $$x_{new} = \frac{x - \mu}{\sigma}$$
* **`MinMaxScaler`:** Compresses or extends all data values so they fit perfectly into a strict bounding range, typically between 0 and 1.
  $$x_{new} = \frac{x - x_{min}}{x_{max} - x_{min}}$$
* **`RobustScaler`:** Natively centers features around the median and scales using the Interquartile Range (IQR). This makes it highly resistant to extreme outliers that could distort standard mean calculations.
  $$x_{new} = \frac{x - \text{median}}{Q_3 - Q_1}$$
* **`Normalizer`:** Scales each individual sample row independently so that its feature vector has a Euclidean length (L2 norm) of exactly 1. This is specifically useful when the direction (or angle) of the feature vector matters more than its absolute magnitude, which is common in text processing text and document classification.

In [ ]:
# Load the real-world high-dimensional Breast Cancer dataset to demonstrate preprocessing scaling
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state=1)

print(f"Original training feature space bounds: Min={X_train.min():.2f}, Max={X_train.max():.2f}")

# Instantiate and fit the MinMaxScaler
minmax_scaler = MinMaxScaler()
minmax_scaler.fit(X_train)

# Apply the fitted scaling parameters transformation
X_train_scaled = minmax_scaler.transform(X_train)
X_test_scaled = minmax_scaler.transform(X_test)

print(f"Scaled training feature space bounds:   Min={X_train_scaled.min():.2f}, Max={X_train_scaled.max():.2f}")
print(f"Scaled testing feature space bounds:    Min={X_test_scaled.min():.2f}, Max={X_test_scaled.max():.2f}")

print("\nCrucial Lifecycle Note: The testing set was explicitly transformed using parameters fitted on the training set.")
print("Never call fit or fit_transform on the testing split, as this introduces severe data leakage errors.")

## 3.3 Dimensionality Reduction: Principal Component Analysis (PCA)
**Principal Component Analysis (PCA)** is an unsupervised data transformation technique used to compress a high-dimensional feature space down into a lower-dimensional representation, while retaining as much variance as possible.

### Algorithmic Mechanics
PCA works by identifying the direction of maximum variance in the data and rotating the coordinate space along it. This newly created axis is called **Principal Component 1 (PC1)**. 

The algorithm then finds the next direction of maximum remaining variance that is strictly perpendicular (orthogonal) to the first axis, creating **Principal Component 2 (PC2)**. This process repeats for all dimensions. By discarding components that contribute little to the total variance, we can compress high-dimensional datasets into 2D or 3D spaces for clear visual analysis.

In [ ]:
# PCA requires standard normal preprocessing scaling before execution
standard_scaler = StandardScaler()
X_scaled_cancer = standard_scaler.fit_transform(cancer.data)

# Instantiate PCA to reduce the 30 features down to the top 2 Principal Components
pca = PCA(n_components=2)
X_pca_cancer = pca.fit_transform(X_scaled_cancer)

print(f"Original feature array dimensionality: {X_scaled_cancer.shape}")
print(f"PCA reduced feature space dimensionality: {X_pca_cancer.shape}")

# Plot the compressed dimensions using a scatter plot matrix interface
plt.figure(figsize=(8, 6))
classes = ['Malignant', 'Benign']
scatter = plt.scatter(X_pca_cancer[:, 0], X_pca_cancer[:, 1], c=cancer.target, cmap='viridis', alpha=0.6, edgecolors='none')
plt.title("PCA Dimension Reduction: Breast Cancer Dataset (30D down to 2D)", fontsize=12)
plt.xlabel("First Principal Component (PC1)")
plt.ylabel("Second Principal Component (PC2)")
plt.legend(handles=scatter.legend_elements()[0], labels=classes)
plt.show()

print("PCA Performance Analysis:")
print(f"The cumulative explained variance ratio of the first two PCs is {np.sum(pca.explained_variance_ratio_)*100:.1f}%")
print("Notice how the Benign and Malignant target classes organically split along the PC1 horizontal axis.")

## 3.4 Clustering Algorithms
Clustering is the task of grouping a dataset into distinct partitions where data points in the same group are highly similar, while points in different groups are distinct.

### 3.4.1 K-Means Clustering
K-Means is one of the simplest and most frequently used clustering algorithms. It iteratively partitions data by minimizing the distance between each point and its nearest cluster center.

#### The Step-by-Step Cycle:
1. Pick a pre-defined number of clusters ($K$) and randomly place $K$ cluster centers (**centroids**) into the multi-dimensional feature space.
2. **Assignment Step:** Assign each individual data point to its closest centroid using Euclidean distance calculations.
3. **Update Step:** Recalculate each centroid's position by taking the exact mean location of all data points currently assigned to that cluster.
4. Repeat steps 2 and 3 until the centroids stabilize and stop moving.

#### Core Failures and Structural Disadvantages:
K-Means relies on simple distance calculations from a central point, which introduces two major limitations:
1. It assumes that clusters have a uniform, spherical shape. It fails completely when handling complex geometries, non-convex boundaries, elongated groupings, or elongated shapes.
2. It cannot natively determine the optimal number of clusters on its own. The developer must manually test and define the target value ($K$) upfront.

In [ ]:
# Replicate the book's demonstration of K-Means performance and visual boundary evaluation
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

# Initialize K-Means to identify 3 groups
kmeans = KMeans(n_clusters=3, n_init='auto', random_state=42)
kmeans_labels = kmeans.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot the baseline correct allocations
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='Accent', alpha=0.7)
axes[0].set_title("Ground Truth Visual Distributions")

# Plot the K-Means clustering outcomes alongside calculated centroids
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_labels, cmap='Accent', alpha=0.7)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=250, marker='X', color='red', label='Centroids')
axes[1].set_title("K-Means Identified Clusters")
axes[1].legend()
plt.show()

### 3.4.2 Agglomerative Hierarchical Clustering
Agglomerative clustering creates a hierarchical tree of cluster relationships using a bottom-up merging process. Unlike K-Means, it does not require you to specify a fixed number of clusters upfront.

#### Agglomerative Processing Cycle:
1. Initialize the algorithm by treating every single individual data point as its own isolated, single-point cluster.
2. Identify the two closest clusters according to a chosen **Linkage Criteria** and merge them into a single joint cluster.
3. Repeat step 2 iteratively until all data points have been merged into one giant global cluster.

#### Linkage Criteria Types in scikit-learn:
* **`ward`:** The default option. It minimizes the total squared distance variance within the clusters being merged. This strategy yields highly balanced, evenly sized, spherical clusters.
* **`average`:** Merges the two clusters that have the lowest average distance between all pairs of their data points.
* **`complete`:** Merges the two clusters that have the smallest maximum distance between any single pair of their data points.

In [ ]:
# Demonstrate Agglomerative Clustering performance on synthetic structures
agglom = AgglomerativeClustering(n_clusters=3, linkage='ward')
agglom_labels = agglom.fit_predict(X_blobs)

plt.figure(figsize=(7, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=agglom_labels, cmap='tab10', alpha=0.7)
plt.title("Agglomerative Hierarchical Clustering Result (Ward Linkage)", fontsize=11)
plt.show()

### 3.4.3 DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
**DBSCAN** is a powerful density-based clustering algorithm designed to identify clusters of arbitrary shapes and automatically handle data noise and outliers.

#### Foundational Parameters:
* `eps` (Epsilon): The maximum spatial radius distance used to evaluate neighborhoods around any given point.
* `min_samples`: The minimum number of neighboring data points required within the epsilon radius to define that region as a dense core.

#### Data Point Classifications:
* **Core Points:** Points that have at least `min_samples` within their `eps` radius boundary.
* **Border Points:** Points that have fewer than `min_samples` within their own neighborhood, but are located close to a designated core point.
* **Noise Points:** Outlier points that do not qualify as core or border points. DBSCAN automatically flags these by assigning them a label of `-1`.

#### Advantages:
DBSCAN does not require you to specify the number of clusters upfront. It can trace complex, non-linear geometries (like interlocking rings or moons) that would cause K-Means or Agglomerative clustering to fail completely.

In [ ]:
# Generate a complex, non-linear interlocking moon geometry where distance-based metrics fail
X_moons, y_moons = make_moons(n_samples=250, noise=0.05, random_state=42)

# Scale features using standard normalization before clustering
X_moons_scaled = StandardScaler().fit_transform(X_moons)

# Compare K-Means vs. DBSCAN performance side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Run K-Means
km_moon = KMeans(n_clusters=2, n_init='auto', random_state=42)
axes[0].scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=km_moon.fit_predict(X_moons_scaled), cmap='coolwarm', alpha=0.7)
axes[0].set_title("K-Means Performance Failure on Complex Moon Geometries")

# Run DBSCAN
dbscan_moon = DBSCAN(eps=0.3, min_samples=5)
axes[1].scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=dbscan_moon.fit_predict(X_moons_scaled), cmap='coolwarm', alpha=0.7)
axes[1].set_title("DBSCAN Performance Success identifying Density-Based Chains")
plt.show()

## 3.5 Evaluating Unsupervised Clustering Models
As noted throughout the chapter, scoring unsupervised clustering models can be exceptionally difficult due to the complete absence of target classes.

### 3.5.1 Unsupervised Metrics: Silhouette Coefficient
When target classes are entirely unknown, we use internal clustering validation metrics like the **Silhouette Score**. It measures how close each data point is to points in its assigned cluster compared to points in the next nearest cluster. 

The score ranges from **-1 to +1**:
* A score near **+1** indicates that data points are tightly packed within their own clusters and well-separated from neighboring groups.
* A score near **0** indicates overlapping clusters.
* A negative score indicates that points have been assigned to the wrong clusters entirely.

### 3.5.2 Supervised Metrics: Adjusted Rand Index (ARI)
If you are developing or testing a new clustering workflow and have access to ground-truth labels ($y$), you can use supervised metrics like the **Adjusted Rand Index (ARI)**. 

ARI evaluates the similarity between the predicted cluster assignments and the ground-truth classes. It completely ignores permutation differences (e.g., if a model swaps cluster label IDs `0` and `1`, ARI still recognizes it as a perfect match). ARI yields a score between **0 and 1**, where 1 indicates a perfect match between assignments and labels.

In [ ]:
# Calculate evaluation metrics for our synthetic moon clustering outcomes
km_predictions = km_moon.fit_predict(X_moons_scaled)
db_predictions = dbscan_moon.fit_predict(X_moons_scaled)

print("--- Advanced Clustering Evaluation Scores ---")
print(f"K-Means Silhouette Internal Validation Score: {silhouette_score(X_moons_scaled, km_predictions):.3f}")
print(f"DBSCAN  Silhouette Internal Validation Score: {silhouette_score(X_moons_scaled, db_predictions):.3f}")

print(f"\nK-Means Ground-Truth Adjusted Rand Index (ARI): {adjusted_rand_score(y_moons, km_predictions):.3f}")
print(f"DBSCAN  Ground-Truth Adjusted Rand Index (ARI): {adjusted_rand_score(y_moons, db_predictions):.3f}")

### 3.6 Summary of Key Takeaways
Chapter 3 introduces the foundational techniques required to explore, clean, and cluster data without relying on explicit target variables.

**Core Takeaways:**
1. **Preprocessing is Mandatory:** Distance-based models like K-Means and DBSCAN require uniform feature scaling (via tools like `StandardScaler` or `MinMaxScaler`) to perform accurately.
2. **Dimensionality Reduction:** `PCA` compresses complex, high-dimensional datasets into fewer principal axes, simplifying data visualization and filtering out noise.
3. **Match Algorithms to Data Geometry:** K-Means is fast and scales well but assumes simple, spherical clusters. For complex, non-linear geometries, density-based algorithms like `DBSCAN` are much more robust.
4. **Rigorous Evaluation:** Internal metrics like the *Silhouette Coefficient* assess cluster separation when ground truth is unknown, while the *Adjusted Rand Index (ARI)* offers definitive validation when labels are available.